<a href="https://colab.research.google.com/github/Text-Machine/temporal-adapters/blob/train/temporal-sft-train.ipynb" target="_parent\"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

# Notebook for data preperation

In [2]:
from datasets import Dataset, load_dataset

In [3]:
dataset = load_dataset("Kaspar/key_phrases_dataset")

README.md:   0%|          | 0.00/453 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.04M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
# Inspect dataset structure in detail
ex = dataset['train'][0]
print("Type of 'prompt':", type(ex['prompt']))
print("Type of 'completion':", type(ex['completion']))
print()
print("Prompt entry [0]:", ex['prompt'][0])
print("Completion entry [0]:", ex['completion'][0])
print()
print("Keys in prompt[0]:", list(ex['prompt'][0].keys()))
print("Keys in completion[0]:", list(ex['completion'][0].keys()))

Type of 'prompt': <class 'list'>
Type of 'completion': <class 'list'>

Prompt entry [0]: {'content': 'Generate an article published in 1890 given the key phrases [\'John Savage funeral\', \'typhoid fever death\', \'cemetery burial\', \'Rev. Thomas Rigby\', "St. Peter\'s Church sidesman", \'Hesketh Parr accident\', \'bus fall injury\', "picnic from St. Mary\'s Church Bank Quay", \'Warrington boy hurt\', \'head injury treatment\']', 'role': 'user'}
Completion entry [0]: {'content': "THE LATE MR. JOHN SAVAGE.\n\nThe funeral of the late Mr. John Savage took\nplace at the cemetery on Saturday afternoon. The\ndeceased, who was the third son of the late Mr. Wm.\nSavage, of Church-st., was 26 years of age at the\ntime of his death, which was due to typhoid fever.\nThe ceremony at the gravesidc was performed by\nthe Rev. Thomas Rigby, vicar of St. Peter's Church,\nof which the deceased was one of the sidesmen.\nThere was a large attendance. A number of choice\nwreaths had been forwarded.\nACCID

In [ ]:
import trl
print("TRL version:", trl.__version__)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Chat template:
{{ bos_token }}
{%- if messages[0]['role'] == 'system' -%}
    {%- if messages[0]['content'] is string -%}
        {%- set first_user_prefix = messages[0]['content'] + '

' -%}
    {%- else -%}
        {%- set first_user_prefix = messages[0]['content'][0]['text'] + '

' -%}
    {%- endif -%}
    {%- set loop_messages = messages[1:] -%}
{%- else -%}
    {%- set first_user_prefix = "" -%}
    {%- set loop_messages = messages -%}
{%- endif -%}
{%- for message in loop_messages -%}
    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) -%}
        {{ raise_exception("Conversation roles must alternate user/assistant/user/assistant/...") }}
    {%- endif -%}
    {%- if (message['role'] == 'assistant') -%}
        {%- set role = "model" -%}
    {%- else -%}
        {%- set role = message['role'] -%}
    {%- endif -%}
    {{ '<start_of_turn>' + role + '
' + (first_user_prefix if loop.first else "") }}
    {%- if message['content'] is string -%}
        {{ message['conte

## Prepare dataset for SFT

The `prompt` and `completion` columns are **lists of chat-message dicts** (e.g. `[{"role": "user", "content": "..."}]`).
TRL's `SFTTrainer` expects a single `messages` column containing the full conversation.
We merge the two lists into one per example.

In [ ]:
def merge_prompt_completion(example):
    """Merge prompt and completion message lists into a single 'messages' column."""
    messages = example["prompt"] + example["completion"]
    return {"messages": messages}

sft_dataset = dataset["train"].map(merge_prompt_completion, remove_columns=["prompt", "completion"])

# Verify the result
print("Columns:", sft_dataset.column_names)
print(f"Num examples: {len(sft_dataset)}")
print("\nExample conversation:")
for msg in sft_dataset[0]["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:120]}...")

## Install dependencies

In [ ]:
!pip install -q trl peft accelerate bitsandbytes

## Load model and tokenizer

Load `google/gemma-3-1b-it` with 4-bit QLoRA quantization to reduce memory. Configure LoRA adapter targeting the attention projection layers (`q_proj`, `v_proj`).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

model_id = "google/gemma-3-1b-it"

# 4-bit quantization for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Patch the chat template to add {% generation %} / {% endgeneration %} markers
# around assistant content. TRL v0.29+ requires these for assistant_only_loss=True.
CHAT_TEMPLATE = (
    "{{ bos_token }}"
    "{% if messages[0]['role'] == 'system' %}"
    "{% if messages[0]['content'] is string %}"
    "{% set first_user_prefix = messages[0]['content'] + '\n\n' %}"
    "{% else %}"
    "{% set first_user_prefix = messages[0]['content'][0]['text'] + '\n\n' %}"
    "{% endif %}"
    "{% set loop_messages = messages[1:] %}"
    "{% else %}"
    "{% set first_user_prefix = '' %}"
    "{% set loop_messages = messages %}"
    "{% endif %}"
    "{% for message in loop_messages %}"
    "{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}"
    "{{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}"
    "{% endif %}"
    "{% if message['role'] == 'assistant' %}"
    "{% set role = 'model' %}"
    "{% else %}"
    "{% set role = message['role'] %}"
    "{% endif %}"
    "{{ '<start_of_turn>' + role + '\n' + (first_user_prefix if loop.first else '') }}"
    "{% if message['role'] == 'assistant' %}{% generation %}"
    "{% endif %}"
    "{% if message['content'] is string %}"
    "{{ message['content'] | trim }}"
    "{% elif message['content'] is iterable %}"
    "{% for item in message['content'] %}"
    "{% if item['type'] == 'image' %}"
    "{{ '<start_of_image>' }}"
    "{% elif item['type'] == 'text' %}"
    "{{ item['text'] | trim }}"
    "{% endif %}"
    "{% endfor %}"
    "{% else %}"
    "{{ raise_exception('Invalid content type') }}"
    "{% endif %}"
    "{% if message['role'] == 'assistant' %}{% endgeneration %}"
    "{% endif %}"
    "{{ '<end_of_turn>\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<start_of_turn>model\n' }}"
    "{% endif %}"
)

tokenizer.chat_template = CHAT_TEMPLATE
print("Chat template patched with {% generation %} markers")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager",
)

# LoRA adapter config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"Model loaded: {model_id}")
print(f"LoRA rank: {lora_config.r}, alpha: {lora_config.lora_alpha}")

## Configure and run SFT training

Use TRL's `SFTConfig` + `SFTTrainer` with:
- **Completions-only training** via `assistant_only_loss=True` in `SFTConfig` — loss is computed only on assistant (completion) tokens, not on the user prompt. This replaces the removed `DataCollatorForCompletionOnlyLM`.
- The `messages` column is automatically formatted using the model's chat template
- LoRA adapters are applied via the `peft_config` argument

In [ ]:
from trl import SFTTrainer, SFTConfig

output_dir = "./sft-gemma-3-1b-keyphrase"

sft_config = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    max_seq_length=1024,
    # Only compute loss on assistant (completion) tokens, not user prompt tokens.
    # This replaces the removed DataCollatorForCompletionOnlyLM.
    assistant_only_loss=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.model.print_trainable_parameters()

In [ ]:
# Launch training
train_result = trainer.train()
print(f"\nTraining loss: {train_result.training_loss:.4f}")

In [ ]:
# Save the final LoRA adapter and tokenizer
final_adapter_dir = f"{output_dir}/final-adapter"
trainer.model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)
print(f"Adapter saved to: {final_adapter_dir}")